# Productivity analysis

Exploratory analysis of FocusBuddy focus sessions: **when** people focus, **how long** sessions last,
**how often** they run to completion, and what the statistics can and cannot support.

**This notebook contains no analysis logic.** Every number and chart comes from the tested modules in
`analytics/` (`dataset`, `analysis`, `statistics`, `inference`, `charts`). The notebook is the narrative.

**Privacy.** Users appear as ids. The database connection is read-only. Save this notebook *without
outputs* if it was run against production data (`Kernel → Restart & Clear Output`).

**Known data limitations (from Phase 0)**
- Focus logs are written only for the user who *ended* the session (the room owner).
- Break sessions are logged as focus too; the phase is not stored.
- Session outcomes (full / early / abandoned) are *derived*; the app does not record why a session ended.
- Timestamps are stored in UTC; time-of-day analysis converts to Europe/Athens.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from analytics import analysis, charts, inference, statistics
from analytics.dataset import load_dataset
from analytics.quality import engine_from_env

TZ = "Europe/Athens"
pd.set_option("display.precision", 2)

## 1. Load and validate

`load_dataset()` runs one query per table and applies the data-quality rules. The reports show what was
flagged and why; only rows with `quality == "ok"` are analysed.

In [ ]:
engine = engine_from_env()  # DATABASE_URL from .env or the environment, opened read-only
with engine.connect() as conn:
    ds = load_dataset(conn)

for report in ds.reports:
    print(report, end="\n\n")

## 2. How long are sessions?

The mean is pulled up by longer sessions, the median is not. The *balanced* mean gives every user equal
weight, so heavy users do not dominate.

In [ ]:
display(analysis.overall_summary(ds.logs).to_frame("value"))
display(pd.Series(statistics.focus_distribution(ds.logs), name="focused minutes").to_frame())
charts.session_length_distribution(ds.logs)

Sessions cluster at the preset lengths (25 and 50 minutes). That is why Tukey's IQR rule is not used
to delete outliers here: with most values tied, IQR collapses to 0 and every non-25 session looks extreme.

## 3. When does focus happen?

In [ ]:
by_hour = analysis.focus_by_hour(ds.logs, TZ)
charts.focus_by_hour(by_hour, tz=TZ)

In [ ]:
charts.weekday_hour_heatmap(analysis.weekday_hour_matrix(ds.logs, TZ))

In [ ]:
display(analysis.focus_by_weekday(ds.logs, TZ)[["weekday_name", "sessions", "total_minutes", "mean_minutes"]])

## 4. Trend

Days without sessions are filled with 0 *before* the rolling mean, so a 7-day window always spans 7
calendar days.

In [ ]:
charts.focus_over_time(analysis.daily_focus(ds.logs, window=7, tz=TZ))

## 5. Completion

In [ ]:
outcomes = analysis.session_outcomes(ds.sessions, ds.logs)
rate = analysis.completion_rate(outcomes)
display(rate)
charts.completion_breakdown(rate)

## 6. Users and rooms

In [ ]:
users = analysis.user_focus_summary(ds.logs, TZ)
display(users[["user_id", "sessions", "active_days", "total_hours", "mean_minutes", "median_minutes"]].head(10))
charts.user_leaderboard(users)

In [ ]:
rooms = analysis.room_activity(ds.logs, ds.rooms, TZ)
charts.room_size_vs_focus(rooms)

Room size is *today's* member count, not the count at the time of each session, so this scatter is
descriptive only and must not be used as a model feature.

## 7. What do the statistics support?

Each result reports an estimate, a confidence interval and an effect size, and is marked
*not interpreted* when there are fewer than 10 independent observations.

In [ ]:
mean, lo, hi = inference.mean_ci(ds.logs.loc[ds.logs["quality"] == "ok", "focused_seconds"] / 60)
print(f"Mean focused minutes: {mean:.1f} (95% CI [{lo:.1f}, {hi:.1f}])\n")
print(inference.weekend_vs_weekday(ds.logs, unit="user", tz=TZ).summary(), end="\n\n")
print(inference.weekend_vs_weekday(ds.logs, unit="session", tz=TZ).summary(), end="\n\n")
print(inference.activity_vs_session_length(ds.logs).summary())

## 8. Conclusions

Write conclusions only from the cells above, and for each one state:

1. **The number** with its uncertainty (CI), not just a point estimate.
2. **The unit of analysis**: sessions or users. Many sessions from few users is still few users.
3. **What it does not show.** This is observational data: differences between groups are associations,
   not effects of the weekday, the hour or the room.
4. **The data limitations** listed at the top, especially the owner-only logging.